# ARCHIVED · Colab notebook: OST track-level features and early tokenizer

Original file name: `Tokenizare_Songs_from_drive.ipynb` (Google Colab, inputs on Google Drive). Kept unchanged except for this cell.

- **Cells 3–9 (used):** track-level combat/chill features for Witcher 3, saved as `csv/combat_chill_profile_W3_raw.csv` and `csv/combat_chill_profile_W3_normalized.csv` (the combat scores printed by cell 9 match the CSV). The same functions with the Skyrim folder are in `src/archive/compute_combat_chill_profile.py` (→ `csv/combat_chill_profile_normalized_v2.csv`). These CSVs provide the combat-feature percentiles in `configs/norm_stats.json`.
- **Cell 2 (commented out, superseded):** early Skyrim tokenizer, not used for the reported models. It writes `train_*` / `test_*` files with per-track min–max tension, no `combat_score` and no `frame_rate`, which `src/build_manifest.py` would reject. The OST tensors in the manifest (`skyrim_*.pt`, `witcher3_*.pt`) come from a later notebook that is not in the repository; `notebooks/02_control_statistics_audit.ipynb` checks their pipeline from the data.
- These track-level features differ from the window-level combat score of the tokenizer: whole track instead of a 5-second causal window, and `librosa.load` at 22.05 kHz without peak normalisation instead of 24 kHz audio peak-normalised to 0.95.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install torchaudio encodec

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 87.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for encodec: filename=encodec-0.1.1-py3-none-any.whl size=45759 sha256=ef6c41083a78fffe60e63db1c51a1017e40a9833ee9ba53fd6dec165423a611e
  Stored in directory: /root/.cache/pip/wheels/b8/eb/9f/e13610cc46ab39d3199fbabebd1c3e142d44b679526e0f228a
Successfully built encodec


In [ ]:
# import os
# import random
# import shutil
# import torch
# import torchaudio
# from pathlib import Path
# from encodec import EncodecModel
# from encodec.utils import convert_audio

# # ──────────────────────────────────────────
# # 1. Setup & Seeding
# # ──────────────────────────────────────────
# RANDOM_SEED = 42
# random.seed(RANDOM_SEED)
# torch.manual_seed(RANDOM_SEED)

# # Output directory directly in Google Drive
# OUTPUT_DIR = Path("/content/drive/MyDrive/DeepLearningProject/tensors")
# TRAIN_OUT  = OUTPUT_DIR / "train"
# TEST_OUT   = OUTPUT_DIR / "test"
# TRAIN_OUT.mkdir(parents=True, exist_ok=True)
# TEST_OUT.mkdir(parents=True, exist_ok=True)

# # ──────────────────────────────────────────
# # 2. Load EnCodec Model
# # ──────────────────────────────────────────
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(f"Using device: {device}")

# model = EncodecModel.encodec_model_24khz()
# model.set_target_bandwidth(24.0)
# model.to(device)
# model.eval()
# print(f"Model sample rate: {model.sample_rate}, channels: {model.channels}")

# # ──────────────────────────────────────────
# # 3. Fast pre-screening: compute global RMS
# # ──────────────────────────────────────────
# def quick_rms(file_path: Path) -> float:
#     """Load audio, downsample to 8kHz mono for speed, return mean RMS."""
#     try:
#         wav, sr = torchaudio.load(file_path)
#         if wav.shape[0] > 1:
#             wav = wav.mean(dim=0, keepdim=True)
#         if sr != 8000:
#             wav = torchaudio.functional.resample(wav, sr, 8000)
#         return float(wav.pow(2).mean().sqrt())
#     except Exception:
#         return 0.0

# # ────────────────────────────────────────
# # 4. Find and Split Files Dynamically
# # ──────────────────────────────────────────
# # POINT THIS TO THE SINGLE FOLDER IN DRIVE CONTAINING ALL YOUR CUSTOM SONGS + IMAGES
# DATASET_BASE = Path("/content/drive/MyDrive/Robe/Skyrim soundtracks/jeremy-soule-the-elder-scrolls-v-skyrim-the-original-game-soundtrack")

# def get_and_split_audio_files(train_ratio=0.85):
#     if not DATASET_BASE.exists():
#         raise FileNotFoundError(f"Directory not found: {DATASET_BASE}")

#     # 1. Filter out images, get only audio
#     supported_extensions = {".flac", ".mp3", ".wav", ".m4a", ".ogg"}
#     all_audio_files = [
#         f for f in DATASET_BASE.rglob("*")
#         if f.suffix.lower() in supported_extensions
#     ]

#     total_files = len(all_audio_files)
#     if total_files == 0:
#         raise FileNotFoundError(f"No audio files found inside {DATASET_BASE}")

#     print(f"Found {total_files} valid audio files (ignored images/other files).")
#     print(f"Pre-screening {total_files} files for RMS tension...")

#     # 2. Score all files for RMS
#     scored = []
#     for i, f in enumerate(all_audio_files):
#         rms = quick_rms(f)
#         scored.append((rms, f))

#     # 3. Sort by energy to split into chill/hype pools
#     scored.sort(key=lambda x: x[0])
#     mid = total_files // 2
#     chill_pool = [f for _, f in scored[:mid]]
#     hype_pool  = [f for _, f in scored[mid:]]

#     # Shuffle the pools so the selection is random within the energy brackets
#     random.shuffle(chill_pool)
#     random.shuffle(hype_pool)

#     # 4. Calculate exact counts based on ratio
#     chill_train_count = int(len(chill_pool) * train_ratio)
#     hype_train_count  = int(len(hype_pool) * train_ratio)

#     # 5. Build Train and Test lists
#     train_mixes = chill_pool[:chill_train_count] + hype_pool[:hype_train_count]
#     test_mixes  = chill_pool[chill_train_count:] + hype_pool[hype_train_count:]

#     # Final shuffle
#     random.shuffle(train_mixes)
#     random.shuffle(test_mixes)

#     print(f"Dynamically split into: {len(train_mixes)} Train tracks, {len(test_mixes)} Test tracks.")
#     return train_mixes, test_mixes

# # ──────────────────────────────────────────
# # 5. Processing Engine
# # ──────────────────────────────────────────
# def process_track(file_path, save_name):
#     try:
#         track_id = file_path.stem
#         wav, sr = torchaudio.load(file_path)

#         original_samples = wav.shape[-1]
#         duration_sec = original_samples / sr

#         if wav.shape[0] > 1:
#             wav = wav.mean(dim=0, keepdim=True)

#         wav = convert_audio(wav, sr, model.sample_rate, model.channels)

#         peak = wav.abs().max()
#         if peak > 0:
#             wav = wav / peak * 0.95

#         wav = wav.unsqueeze(0).to(device)

#         with torch.no_grad():
#             encoded_frames = model.encode(wav)
#             codes, scales  = encoded_frames[0]
#             tokens = codes[0].cpu()

#         wav_mono   = wav[0].mean(dim=0).cpu()
#         num_samps  = wav_mono.shape[-1]
#         num_frames = tokens.shape[-1]
#         frame_size = num_samps // num_frames if num_frames > 0 else 1

#         pad_len = (-num_samps) % frame_size
#         wav_padded = (
#             torch.nn.functional.pad(wav_mono, (0, pad_len))
#             if pad_len > 0 else wav_mono
#         )

#         rms_frames = torch.sqrt(
#             torch.mean(wav_padded.unfold(-1, frame_size, frame_size) ** 2, dim=-1)
#         )
#         tension = (rms_frames - rms_frames.min()) / (
#             rms_frames.max() - rms_frames.min() + 1e-8
#         )

#         min_len = min(tokens.shape[-1], tension.shape[-1])
#         tokens  = tokens[:, :min_len]
#         tension = tension[:min_len]

#         save_path = (TRAIN_OUT if save_name.startswith("train") else TEST_OUT) / f"{save_name}.pt"
#         torch.save({
#             "track_id":        track_id,
#             "duration_sec":    duration_sec,
#             "original_samples": original_samples,
#             "tokens":          tokens,
#             "tension":         tension,
#             "sample_rate":     model.sample_rate,
#             "bandwidth_kbps":  24.0,
#         }, save_path)

#         del wav, wav_mono, wav_padded, tokens, tension, encoded_frames, codes, scales
#         torch.cuda.empty_cache()
#         return True

#     except Exception as e:
#         print(f"Failed to process {file_path}: {e}")
#         torch.cuda.empty_cache()
#         return False

# # ──────────────────────────────────────────
# # 6. Execution
# # ──────────────────────────────────────────
# print("=== Analyzing Directory and Splitting Data ===")
# train_mixes, test_mixes = get_and_split_audio_files(train_ratio=0.85)

# print(f"\nStarting encoding: {len(train_mixes)} train, {len(test_mixes)} test...")

# train_ok = 0
# for idx, track_path in enumerate(train_mixes):
#     if process_track(track_path, f"train_{idx:03d}"):
#         train_ok += 1
#         if train_ok % 10 == 0:
#             print(f"  Encoded {train_ok}/{len(train_mixes)} train tracks...")

# print(f"Train done: {train_ok}/{len(train_mixes)} succeeded.")

# test_ok = 0
# for idx, track_path in enumerate(test_mixes):
#     if process_track(track_path, f"test_{idx:03d}"):
#         test_ok += 1
#         if test_ok % 5 == 0:
#             print(f"  Encoded {test_ok}/{len(test_mixes)} test tracks...")

# print(f"Test done: {test_ok}/{len(test_mixes)} succeeded.")

# print("Zipping artifacts...")
# shutil.make_archive("/content/drive/MyDrive/custom_vibe_dataset", "zip", OUTPUT_DIR)
# print("Artifact zipped successfully! You can now safely close your PC.")

/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Using device: cpu
Downloading: "https://dl.fbaipublicfiles.com/encodec/v0/encodec_24khz-d7cc33bc.th" to /root/.cache/torch/hub/checkpoints/encodec_24khz-d7cc33bc.th


100%|██████████| 88.9M/88.9M [00:00<00:00, 435MB/s]


Model sample rate: 24000, channels: 1
=== Analyzing Directory and Splitting Data ===
Found 79 valid audio files (ignored images/other files).
Pre-screening 79 files for RMS tension...
Dynamically split into: 67 Train tracks, 12 Test tracks.

Starting encoding: 67 train, 12 test...
  Encoded 10/67 train tracks...
  Encoded 20/67 train tracks...
  Encoded 30/67 train tracks...
  Encoded 40/67 train tracks...
  Encoded 50/67 train tracks...
  Encoded 60/67 train tracks...
Train done: 67/67 succeeded.
  Encoded 5/12 test tracks...
  Encoded 10/12 test tracks...
Test done: 12/12 succeeded.
Zipping artifacts...
Artifact zipped successfully! You can now safely close your PC.


In [ ]:
import librosa
import numpy as np

def compute_percussive_ratio(y, sr):
    """
    Returns fraction of total energy that is percussive.
    High value → combat (strong beat-driven percussion)
    Low value  → chill (mostly harmonic, piano/strings)
    """
    y_harmonic, y_percussive = librosa.effects.hpss(y)
    harm_energy = np.sum(y_harmonic ** 2)
    perc_energy = np.sum(y_percussive ** 2)
    total = harm_energy + perc_energy + 1e-8
    return perc_energy / total

def compute_beat_alignment(y, sr):
    """
    Measures how strongly percussive onsets align with detected beats.
    High → tight, on-the-beat combat rhythm
    Low  → loose, ambient, non-metric chill
    """
    y_harmonic, y_percussive = librosa.effects.hpss(y)
    tempo, beats = librosa.beat.beat_track(y=y_percussive, sr=sr, units='frames')
    onset_env = librosa.onset.onset_strength(y=y_percussive, sr=sr)

    if len(beats) == 0:
        return 0.0

    beat_strengths = onset_env[np.clip(beats, 0, len(onset_env) - 1)]
    return float(np.mean(beat_strengths)) if len(beat_strengths) > 0 else 0.0


def compute_note_duration_stats(y, sr):
    """
    Returns mean inter-onset interval (IOI) in seconds.
    Short IOI → combat (rapid, short notes)
    Long IOI  → chill (sustained, long notes)
    """
    onsets = librosa.onset.onset_detect(y=y, sr=sr, units='time')
    if len(onsets) < 2:
        return None  # too sparse to measure
    iois = np.diff(onsets)
    return float(np.mean(iois))


def compute_dissonance_score(y, sr):
    """
    Rough dissonance proxy: correlation between adjacent chroma bins.
    High simultaneous energy in adjacent semitones → dissonant/tense.
    """
    chroma = librosa.feature.chroma_cqt(y=y, sr=sr)  # [12, T]
    # Sum of energy in semitone-adjacent pairs (proxy for clashing intervals)
    adjacent_energy = np.sum(chroma[:-1] * chroma[1:], axis=0)  # [T]
    total_energy = np.sum(chroma, axis=0) + 1e-8
    dissonance = adjacent_energy / total_energy
    return float(np.mean(dissonance))

def compute_percussive_loudness(y, sr):
    """
    Absolute RMS energy of the percussive component (not a ratio).
    High → heavy drums/hits (combat)
    Low  → gentle strums/plucks (bard/folk)
    """
    y_harmonic, y_percussive = librosa.effects.hpss(y)
    return float(np.sqrt(np.mean(y_percussive ** 2)))

def compute_spectral_centroid_mean(y, sr):
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
    return float(np.mean(centroid))

In [ ]:
def compute_combat_chill_profile(file_path, sr_target=22050):
    y, sr = librosa.load(file_path, sr=sr_target, mono=True)

    perc_ratio   = compute_percussive_ratio(y, sr)
    perc_loud    = compute_percussive_loudness(y, sr)
    beat_align   = compute_beat_alignment(y, sr)   # reference only
    ioi          = compute_note_duration_stats(y, sr)
    dissonance   = compute_dissonance_score(y, sr)
    centroid     = compute_spectral_centroid_mean(y, sr)

    note_shortness = float(np.clip(1.0 - (ioi / 1.5), 0.0, 1.0)) if ioi else 0.0

    return {
        "perc_ratio": perc_ratio,
        "percussive_loudness": perc_loud,
        "beat_align": beat_align,
        "mean_ioi_sec": ioi,
        "note_shortness": note_shortness,
        "dissonance": dissonance,
        "spectral_centroid_mean": centroid,
    }

In [ ]:
def normalize_features_and_rescore(df):
    features = [
        "perc_ratio",
        "percussive_loudness",
        "note_shortness",
        "dissonance",
        "spectral_centroid_mean",
    ]

    df_norm = df.copy()
    for feat in features:
        p5 = df[feat].quantile(0.05)
        p95 = df[feat].quantile(0.95)
        df_norm[feat + "_norm"] = ((df[feat] - p5) / (p95 - p5 + 1e-8)).clip(0, 1)

    df_norm["combat_score"] = (
        0.30 * df_norm["perc_ratio_norm"] +
        0.25 * df_norm["percussive_loudness_norm"] +
        0.10 * df_norm["note_shortness_norm"] +
        0.15 * df_norm["dissonance_norm"] +
        0.20 * df_norm["spectral_centroid_mean_norm"]
    )

    df_norm["chill_score"] = 1.0 - df_norm["combat_score"]
    return df_norm

In [ ]:
def find_audio_files_deduped(folder):
    all_files = find_audio_files(folder)
    by_stem = {}
    for f in all_files:
        key = f.stem.lower()
        if key not in by_stem or f.suffix.lower() == ".flac":
            by_stem[key] = f
    return sorted(by_stem.values())

In [ ]:
from os import path
import pandas as pd
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/Robe/The witcher 3 soundtracks")

AUDIO_EXTENSIONS = {".flac", ".mp3", ".wav", ".ogg"}

def find_audio_files(folder: Path):
    return sorted([
        f for f in folder.rglob("*")
        if f.is_file() and f.suffix.lower() in AUDIO_EXTENSIONS
    ])

# 1. Extract raw features (what you already did)
audio_files = find_audio_files_deduped(DATA_DIR)   # dedupe flac/mp3 first
print(f"Found {len(audio_files)} audio files")

# 1. Extract (re-run needed — percussive_loudness is new)
results = []
for i, fp in enumerate(audio_files):
    try:
        profile = compute_combat_chill_profile(fp)
        profile["track_id"] = fp.stem
        results.append(profile)
        print(f"[{i+1}/{len(audio_files)}] {fp.name} done")
    except Exception as e:
        print(f"[{i+1}/{len(audio_files)}] FAILED: {fp.name} → {e}")

df = pd.DataFrame(results)
df_fixed = normalize_features_and_rescore(df)

Found 35 audio files
[1/35] 01 The Trail.flac done
[2/35] 02 Geralt of Rivia.flac done
[3/35] 03 Eredin, King of the Hunt.flac done
[4/35] 04 Wake Up, Ciri.flac done
[5/35] 05 Aen Seidhe.flac done
[6/35] 06 Commanding the Fury.flac done
[7/35] 07 Emhyr var Emreis.flac done
[8/35] 08 Spikeroog.flac done
[9/35] 09 King Bran Final Voyage.flac done
[10/35] 10 Silver for Monsters....flac done
[11/35] 11 Whispers of Oxenfurt.flac done
[12/35] 12 The Nightingale.flac done
[13/35] 13 City of Intrigues.flac done
[14/35] 14 The Hunter Path.flac done
[15/35] 15 Widow-maker.flac done
[16/35] 16 Kaer Morhen.flac done
[17/35] 17 Eyes of the Wolf.flac done
[18/35] 18 Witch Hunters.flac done
[19/35] 19 ...Steel for Humans.flac done
[20/35] 20 Fate Calls.flac done
[21/35] 21 The Vagabond.flac done
[22/35] 22 Cloak and Dagger.flac done
[23/35] 23 Forged in Fire.flac done
[24/35] 24 Yes, I Do....flac done
[25/35] 25 Welcome, Imlerith.flac done
[26/35] 26 Drink Up, There’s More!.flac done
[27/35] 27 After

In [ ]:
from pathlib import Path

SAVE_DIR = Path("/content/drive/MyDrive/DeepLearningProject")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Save the raw (pre-normalization) features
df.to_csv(SAVE_DIR / "combat_chill_profile_Witcher3_raw.csv", index=False)

# Save the normalized, scored version
df_fixed.to_csv(SAVE_DIR / "combat_chill_profile_Witcher3_normalized.csv", index=False)

print(f"Saved raw features to: {SAVE_DIR / 'combat_chill_profile_Witcher3_raw.csv'}")
print(f"Saved normalized scores to: {SAVE_DIR / 'combat_chill_profile_Witcher3_normalized.csv'}")

Saved raw features to: /content/drive/MyDrive/DeepLearningProject/combat_chill_profile_Witcher3_raw.csv
Saved normalized scores to: /content/drive/MyDrive/DeepLearningProject/combat_chill_profile_Witcher3_normalized.csv


In [ ]:
# If df is still in memory from the run, just do:
df_fixed = normalize_features_and_rescore(df)
print(df_fixed.sort_values("combat_score", ascending=False)[["track_id", "combat_score", "chill_score"]].to_string(index=False))

                       track_id  combat_score  chill_score
         19 ...Steel for Humans      0.971198     0.028802
                 15 Widow-maker      0.924275     0.075725
          31 The Hunt is Coming      0.871270     0.128730
     26 Drink Up, There’s More!      0.847208     0.152792
           13 City of Intrigues      0.844377     0.155623
30 The Song of the Sword–Dancer      0.833231     0.166769
      10 Silver for Monsters...      0.827896     0.172104
           35 Hunt or Be Hunted      0.795596     0.204404
         06 Commanding the Fury      0.792456     0.207544
                   01 The Trail      0.784458     0.215542
            17 Eyes of the Wolf      0.745424     0.254576
             12 The Nightingale      0.735382     0.264618
                  05 Aen Seidhe      0.705915     0.294085
              23 Forged in Fire      0.684184     0.315816
           25 Welcome, Imlerith      0.669062     0.330938
   28 Blood on the Cobblestones      0.663458     0.3365